**Task 1 — Import Libraries**

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    LogisticRegression
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Libraries imported successfully.")

**Task 2 — Load cleaned_data.csv**

In [ ]:
df = pd.read_csv("cleaned_data.csv")

print("First five rows:")
display(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

**Task 3 — Define Features and Labels**

In [ ]:
# Regression target
y_reg = df["cholesterol"].copy()

# Natural binary classification target
y_clf = df["heart_attack_risk"].astype(int).copy()

# Remove both targets and patient identifier from features
X = df.drop(
    columns=[
        "cholesterol",
        "heart_attack_risk",
        "patient_id"
    ],
    errors="ignore"
).copy()

print("Feature matrix shape:", X.shape)
print("Regression label:", y_reg.name)
print("Classification label:", y_clf.name)

print("\nClassification target values:")
print(y_clf.value_counts())

**Task 4 — Encode Categorical Columns**

In [ ]:
# Recreate X from the original loaded DataFrame
# This removes the NaN values created by the earlier failed mapping
X = df.drop(
    columns=[
        "cholesterol",
        "heart_attack_risk",
        "patient_id"
    ],
    errors="ignore"
).copy()

print("Physical activity values before cleaning:")
print(X["physical_activity"].value_counts(dropna=False))

# Clean all text columns
text_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in text_columns:
    X[col] = (
        X[col]
        .astype("string")
        .str.strip()
    )

# Fill actual missing physical_activity values with mode
physical_activity_mode = (
    X["physical_activity"]
    .mode(dropna=True)
    .iloc[0]
)

X["physical_activity"] = (
    X["physical_activity"]
    .fillna(physical_activity_mode)
)

# Ordered encoding: Low < Moderate < High
physical_activity_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

X["physical_activity"] = (
    X["physical_activity"]
    .map(physical_activity_mapping)
    .astype(int)
)

print("\nPhysical activity values after mapping:")
print(X["physical_activity"].value_counts(dropna=False))

# Find remaining unordered categorical columns
nominal_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("\nColumns for one-hot encoding:")
print(nominal_columns)

# One-hot encode nominal categories
X_encoded = pd.get_dummies(
    X,
    columns=nominal_columns,
    drop_first=True,
    dtype=int
)

print("\nEncoded feature matrix shape:")
print(X_encoded.shape)

print("\nRemaining non-numeric columns:")
print(
    X_encoded
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("\nMissing values after encoding:")
print(X_encoded.isnull().sum().sum())

print("\nEncoding completed successfully.")

**Task 5 — Leak-Free Train-Test Split**

In [ ]:
(
    X_train,
    X_test,
    y_reg_train,
    y_reg_test,
    y_clf_train,
    y_clf_test
) = train_test_split(
    X_encoded,
    y_reg,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)

print("Regression training labels:", y_reg_train.shape)
print("Regression test labels:", y_reg_test.shape)

print("Classification training labels:", y_clf_train.shape)
print("Classification test labels:", y_clf_test.shape)

**Task 6 — Leak-Free Standard Scaling**

In [ ]:
scaler = StandardScaler()

# Fit only on training features
scaler.fit(X_train)

# Transform training and test features
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)
print(
    "Average training-feature mean after scaling:",
    round(X_train_scaled.mean(), 6)
)

print(
    "Average training-feature standard deviation:",
    round(X_train_scaled.std(), 6)
)

**Task 7 — Linear Regression**

In [ ]:
linear_model = LinearRegression()

linear_model.fit(
    X_train_scaled,
    y_reg_train
)

y_pred_reg = linear_model.predict(
    X_test_scaled
)
linear_mse = mean_squared_error(
    y_reg_test,
    y_pred_reg
)

linear_r2 = r2_score(
    y_reg_test,
    y_pred_reg
)

print("Linear Regression MSE:", round(linear_mse, 4))
print("Linear Regression R²:", round(linear_r2, 4))

linear_coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": linear_model.coef_
})

linear_coefficients[
    "Absolute Coefficient"
] = linear_coefficients[
    "Coefficient"
].abs()

linear_coefficients = linear_coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
).reset_index(drop=True)

print("Linear Regression coefficients:")
display(linear_coefficients)

top_three_linear_features = linear_coefficients.head(3)

print(
    "Three features with the largest "
    "absolute coefficients:"
)

display(top_three_linear_features)
linear_coefficients.to_csv(
    "results/linear_regression_coefficients.csv",
    index=False
)

**Task 8 — Ridge Regression**

In [ ]:
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(
    X_train_scaled,
    y_reg_train
)

y_pred_ridge = ridge_model.predict(
    X_test_scaled
)
ridge_mse = mean_squared_error(
    y_reg_test,
    y_pred_ridge
)

ridge_r2 = r2_score(
    y_reg_test,
    y_pred_ridge
)

print("Ridge Regression MSE:", round(ridge_mse, 4))
print("Ridge Regression R²:", round(ridge_r2, 4))
regression_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression"
    ],
    "MSE": [
        linear_mse,
        ridge_mse
    ],
    "R²": [
        linear_r2,
        ridge_r2
    ]
})

print("Regression model comparison:")
display(regression_comparison)
regression_comparison.to_csv(
    "results/regression_comparison.csv",
    index=False
)
ridge_coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Ridge Coefficient": ridge_model.coef_
})

ridge_coefficients[
    "Absolute Ridge Coefficient"
] = ridge_coefficients[
    "Ridge Coefficient"
].abs()

ridge_coefficients = ridge_coefficients.sort_values(
    "Absolute Ridge Coefficient",
    ascending=False
).reset_index(drop=True)

display(ridge_coefficients.head(10))

**Task 9 — Check Classification Class Imbalance**

In [ ]:
class_count_before = y_clf_train.value_counts().sort_index()

class_percentage_before = (
    y_clf_train.value_counts(normalize=True)
    .sort_index()
    * 100
)

class_balance_before = pd.DataFrame({
    "Count": class_count_before,
    "Percentage": class_percentage_before
})

print("Training-class distribution before balancing:")
display(class_balance_before)
minority_percentage = class_percentage_before.min()

print(
    "Minority-class percentage:",
    round(minority_percentage, 2),
    "%"
)
if minority_percentage < 35:

    print(
        "Class imbalance detected. "
        "Applying SMOTE to the training set only."
    )

    smote = SMOTE(
        random_state=42
    )

    X_clf_train_model, y_clf_train_model = (
        smote.fit_resample(
            X_train_scaled,
            y_clf_train
        )
    )

    imbalance_strategy = "SMOTE"

else:

    print(
        "No severe class imbalance detected. "
        "SMOTE is not required."
    )

    X_clf_train_model = X_train_scaled
    y_clf_train_model = y_clf_train

    imbalance_strategy = "No resampling required"
    class_count_after = pd.Series(
    y_clf_train_model
).value_counts().sort_index()

balance_comparison = pd.DataFrame({
    "Before SMOTE": class_count_before,
    "After SMOTE": class_count_after
}).fillna(0).astype(int)

print("Class-count comparison:")
display(balance_comparison)
balance_comparison.to_csv(
    "results/class_balance_comparison.csv"
)

**Task 10 — Baseline Logistic Regression, C=1.0**

In [ ]:
logistic_baseline = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

logistic_baseline.fit(
    X_clf_train_model,
    y_clf_train_model
)
y_pred_clf_baseline = logistic_baseline.predict(
    X_test_scaled
)

y_proba_baseline = logistic_baseline.predict_proba(
    X_test_scaled
)[:, 1]
baseline_confusion_matrix = confusion_matrix(
    y_clf_test,
    y_pred_clf_baseline
)

print("Confusion matrix:")
print(baseline_confusion_matrix)
plt.figure(figsize=(6, 5))

sns.heatmap(
    baseline_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False
)

plt.title(
    "Logistic Regression Confusion Matrix"
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()

plt.savefig(
    "plots/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
print("Classification report:")

print(
    classification_report(
        y_clf_test,
        y_pred_clf_baseline,
        digits=4
    )
)
baseline_accuracy = accuracy_score(
    y_clf_test,
    y_pred_clf_baseline
)

baseline_precision = precision_score(
    y_clf_test,
    y_pred_clf_baseline,
    zero_division=0
)

baseline_recall = recall_score(
    y_clf_test,
    y_pred_clf_baseline,
    zero_division=0
)

baseline_f1 = f1_score(
    y_clf_test,
    y_pred_clf_baseline,
    zero_division=0
)

baseline_auc = roc_auc_score(
    y_clf_test,
    y_proba_baseline
)

baseline_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "AUC"
    ],
    "Value": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_auc
    ]
})

print("Baseline Logistic Regression metrics:")
display(baseline_metrics)

In [ ]:
# Predictions
y_pred = logistic_baseline.predict(X_test_scaled)
y_prob = logistic_baseline.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_clf_test, y_pred)

print("Confusion Matrix")
print(cm)

# Individual Metrics
accuracy = accuracy_score(y_clf_test, y_pred)
precision = precision_score(y_clf_test, y_pred)
recall = recall_score(y_clf_test, y_pred)
f1 = f1_score(y_clf_test, y_pred)
auc = roc_auc_score(y_clf_test, y_prob)

print("\nModel Performance")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC Score: {auc:.4f}")

print("\nClassification Report")
print(classification_report(y_clf_test, y_pred))
metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC Score"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]
})

print("\nOverall Performance Metrics")
display(metrics_df)

**Task 11 — ROC Curve and AUC**

In [ ]:
fpr_baseline, tpr_baseline, thresholds_roc = roc_curve(
    y_clf_test,
    y_proba_baseline
)
plt.figure(figsize=(8, 6))

plt.plot(
    fpr_baseline,
    tpr_baseline,
    label=f"Logistic Regression AUC = {baseline_auc:.4f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curve — Baseline Logistic Regression"
)

plt.legend(loc="lower right")

plt.text(
    0.60,
    0.20,
    f"AUC = {baseline_auc:.4f}",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "plots/roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


**Task 12 — Decision-Threshold Sensitivity**

In [ ]:
decision_thresholds = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70
]

threshold_results = []

for threshold in decision_thresholds:

    threshold_predictions = (
        y_proba_baseline >= threshold
    ).astype(int)

    threshold_precision = precision_score(
        y_clf_test,
        threshold_predictions,
        zero_division=0
    )

    threshold_recall = recall_score(
        y_clf_test,
        threshold_predictions,
        zero_division=0
    )

    threshold_f1 = f1_score(
        y_clf_test,
        threshold_predictions,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": threshold_precision,
        "Recall": threshold_recall,
        "F1": threshold_f1
    })

threshold_table = pd.DataFrame(
    threshold_results
)

print("Decision-threshold comparison:")
display(threshold_table)
best_threshold_row = threshold_table.loc[
    threshold_table["F1"].idxmax()
]

best_f1_threshold = best_threshold_row[
    "Threshold"
]

best_threshold_f1 = best_threshold_row[
    "F1"
]

print(
    "Threshold with maximum F1-score:",
    best_f1_threshold
)

print(
    "Maximum F1-score:",
    round(best_threshold_f1, 4)
)
threshold_table.to_csv(
    "results/threshold_comparison.csv",
    index=False
)


**Task 13 — Strongly Regularized Logistic Regression, C=0.01**

In [ ]:
logistic_strong_reg = LogisticRegression(
    C=0.01,
    max_iter=1000,
    random_state=42
)

logistic_strong_reg.fit(
    X_clf_train_model,
    y_clf_train_model
)
y_pred_strong_reg = logistic_strong_reg.predict(
    X_test_scaled
)

y_proba_strong_reg = logistic_strong_reg.predict_proba(
    X_test_scaled
)[:, 1]
strong_reg_precision = precision_score(
    y_clf_test,
    y_pred_strong_reg,
    zero_division=0
)

strong_reg_recall = recall_score(
    y_clf_test,
    y_pred_strong_reg,
    zero_division=0
)

strong_reg_f1 = f1_score(
    y_clf_test,
    y_pred_strong_reg,
    zero_division=0
)

strong_reg_auc = roc_auc_score(
    y_clf_test,
    y_proba_strong_reg
)

logistic_regularization_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression C=1.0",
        "Logistic Regression C=0.01"
    ],
    "Precision": [
        baseline_precision,
        strong_reg_precision
    ],
    "Recall": [
        baseline_recall,
        strong_reg_recall
    ],
    "F1-score": [
        baseline_f1,
        strong_reg_f1
    ],
    "AUC": [
        baseline_auc,
        strong_reg_auc
    ]
})

print("Logistic regularization comparison:")
display(logistic_regularization_comparison)
logistic_regularization_comparison.to_csv(
    "results/logistic_regularization_comparison.csv",
    index=False
)

**Task 14 — Bootstrap Confidence Interval for AUC Difference**

In [ ]:
# AUC of C=1.0 model − AUC of C=0.01 model
y_clf_test_array = np.asarray(
    y_clf_test
)

baseline_probability_array = np.asarray(
    y_proba_baseline
)

strong_reg_probability_array = np.asarray(
    y_proba_strong_reg
)
np.random.seed(42)

n_bootstrap = 500
auc_differences = []

attempts = 0

while len(auc_differences) < n_bootstrap:

    attempts += 1

    bootstrap_indices = np.random.choice(
        len(y_clf_test_array),
        size=len(y_clf_test_array),
        replace=True
    )

    bootstrap_y = y_clf_test_array[
        bootstrap_indices
    ]

    # AUC cannot be calculated when only one
    # class appears in a bootstrap sample.
    if len(np.unique(bootstrap_y)) < 2:
        continue

    bootstrap_baseline_probability = (
        baseline_probability_array[
            bootstrap_indices
        ]
    )

    bootstrap_strong_probability = (
        strong_reg_probability_array[
            bootstrap_indices
        ]
    )

    bootstrap_baseline_auc = roc_auc_score(
        bootstrap_y,
        bootstrap_baseline_probability
    )

    bootstrap_strong_auc = roc_auc_score(
        bootstrap_y,
        bootstrap_strong_probability
    )

    auc_difference = (
        bootstrap_baseline_auc
        - bootstrap_strong_auc
    )

    auc_differences.append(
        auc_difference
    )

print("Valid bootstrap samples:", len(auc_differences))
print("Total sampling attempts:", attempts)
auc_differences = np.array(
    auc_differences
)

mean_auc_difference = np.mean(
    auc_differences
)

lower_auc_difference = np.percentile(
    auc_differences,
    2.5
)

upper_auc_difference = np.percentile(
    auc_differences,
    97.5
)

print(
    "Mean AUC difference:",
    round(mean_auc_difference, 6)
)

print(
    "95% confidence interval:",
    (
        round(lower_auc_difference, 6),
        round(upper_auc_difference, 6)
    )
)
if (
    lower_auc_difference > 0
    or upper_auc_difference < 0
):

    confidence_interpretation = (
        "The confidence interval excludes zero. "
        "The difference is likely consistent."
    )

else:

    confidence_interpretation = (
        "The confidence interval includes zero. "
        "The difference may not be reliable."
    )

print(confidence_interpretation)
bootstrap_summary = pd.DataFrame({
    "Metric": [
        "Mean AUC Difference",
        "2.5th Percentile",
        "97.5th Percentile"
    ],
    "Value": [
        mean_auc_difference,
        lower_auc_difference,
        upper_auc_difference
    ]
})

display(bootstrap_summary)

bootstrap_summary.to_csv(
    "results/bootstrap_auc_difference.csv",
    index=False
)

**Task 15 — Save All Main Results**

In [ ]:
summary_results = pd.DataFrame({
    "Metric": [
        "Linear Regression MSE",
        "Linear Regression R²",
        "Ridge Regression MSE",
        "Ridge Regression R²",
        "Baseline Accuracy",
        "Baseline Precision",
        "Baseline Recall",
        "Baseline F1",
        "Baseline AUC",
        "Strong Regularization Precision",
        "Strong Regularization Recall",
        "Strong Regularization F1",
        "Strong Regularization AUC",
        "Best F1 Threshold",
        "Mean Bootstrap AUC Difference",
        "Bootstrap Lower 95%",
        "Bootstrap Upper 95%"
    ],
    "Value": [
        linear_mse,
        linear_r2,
        ridge_mse,
        ridge_r2,
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_auc,
        strong_reg_precision,
        strong_reg_recall,
        strong_reg_f1,
        strong_reg_auc,
        best_f1_threshold,
        mean_auc_difference,
        lower_auc_difference,
        upper_auc_difference
    ]
})

display(summary_results)

summary_results.to_csv(
    "results/part2_summary_results.csv",
    index=False
)


Task 16 — Check Generated Files **bold text**

In [ ]:
print("Plot files:")
print(os.listdir("plots"))

print("\nResult files:")
print(os.listdir("results"))
import shutil
from google.colab import files

shutil.make_archive(
    "part2_plots",
    "zip",
    "plots"
)

shutil.make_archive(
    "part2_results",
    "zip",
    "results"
)

files.download("part2_plots.zip")
files.download("part2_results.zip")
